
# Souled Store Demand Forecasting — Location × SKU

## 1. Project Objective

The goal is to forecast the next 4 weeks of Net Sold Quantity for each fulfilment location and SKU.

- **Dataset:** Souled Store
- **Forecast level:** Fulfilment Location × SKU Code
- **Time interval:** Weekly
- **Target variable:** Net Sold Quantity
- **Forecast horizon:** 4 weeks

We will evaluate simple baseline models and generate a 4-week forecast.


In [ ]:

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Locate the project and processed data folders
project_root = Path.cwd().parent
processed_folder = project_root / "data" / "processed"

# Load Souled Store weekly-filled data
file_path = (
    processed_folder /
    "thesouledstore_Oder_data_cleaned_weekly_filled.csv"
)

df = pd.read_csv(file_path, low_memory=False)

# Convert Week to datetime
df["Week"] = pd.to_datetime(df["Week"])

print("Dataset shape:", df.shape)
print("Date range:", df["Week"].min(), "to", df["Week"].max())
print("Locations:", df["Fulfilment Location"].nunique())
print("SKUs:", df["SKU Code"].nunique())
print("Missing values:")
print(df.isnull().sum())


In [ ]:

evaluation_results = []

for (loc, sku), group in df.groupby(
    ["Fulfilment Location", "SKU Code"]
):
    group = group.sort_values("Week")
    values = group["Net Sold Quantity"].to_list()

    if len(values) < 8:
        continue

    actual = []
    naive_predictions = []
    average_predictions = []

    # Test on the last 4 weeks
    for i in range(len(values) - 4, len(values)):
        actual.append(values[i])
        naive_predictions.append(values[i - 1])
        average_predictions.append(
            sum(values[i-4:i]) / 4
        )

    naive_mae = sum(
        abs(a - p)
        for a, p in zip(actual, naive_predictions)
    ) / 4

    average_mae = sum(
        abs(a - p)
        for a, p in zip(actual, average_predictions)
    ) / 4

    evaluation_results.append({
        "Location": loc,
        "SKU": sku,
        "Weeks": len(values),
        "Naive MAE": naive_mae,
        "4-Week Average MAE": average_mae
    })

evaluation_df = pd.DataFrame(evaluation_results)

print("Combinations evaluated:", len(evaluation_df))
print("\nAverage MAE:")
print(
    evaluation_df[["Naive MAE", "4-Week Average MAE"]]
    .mean()
    .round(2)
)

print("\nNaive model wins:",
      (evaluation_df["Naive MAE"] <
       evaluation_df["4-Week Average MAE"]).sum())

print("Four-week-average model wins:",
      (evaluation_df["4-Week Average MAE"] <
       evaluation_df["Naive MAE"]).sum())



## 4. Generate 4-Week SKU Forecasts

We will use the four-week-average baseline to forecast Net Sold Quantity for each fulfilment location and SKU for the next 4 weeks.


In [ ]:

# Generate forecasts using a common set of future weeks

output_folder = project_root / "outputs"
output_folder.mkdir(parents=True, exist_ok=True)

forecast_start = df["Week"].max() + pd.Timedelta(days=7)

future_weeks = pd.date_range(
    start=forecast_start,
    periods=4,
    freq="7D"
)

forecast_rows = []

for (loc, sku), group in df.groupby(
    ["Fulfilment Location", "SKU Code"]
):
    group = group.sort_values("Week")
    values = group["Net Sold Quantity"].to_list()

    # Use the average of the latest 4 observed weeks
    recent_values = values[-4:]
    prediction = sum(recent_values) / len(recent_values)

    for week in future_weeks:
        forecast_rows.append({
            "Week": week,
            "Fulfilment Location": loc,
            "SKU Code": sku,
            "Forecast Net Sold Quantity": prediction
        })

forecast_df = pd.DataFrame(forecast_rows)

# Save the forecast
forecast_path = (
    output_folder / "souledstore_sku_4_week_forecast.csv"
)
forecast_df.to_csv(forecast_path, index=False)

print("Forecast rows:", len(forecast_df))
print("Unique location-SKU combinations:",
      forecast_df.groupby(
          ["Fulfilment Location", "SKU Code"]
      ).ngroups)
print("Forecast weeks:", sorted(forecast_df["Week"].unique()))

print("\nFirst 10 rows:")
print(forecast_df.head(10).to_string(index=False))

print("\nSaved to:", forecast_path)


In [ ]:
cleaned_path = processed_folder / "thesouledstore_Oder_data_cleaned.csv"
raw_df = pd.read_csv(cleaned_path, low_memory=False)

print("Columns:")
print(raw_df.columns.tolist())

for col in ["Category1", "Sub Category", "Segment"]:
    if col in raw_df.columns:
        print(f"\n{col} — unique values:")
        print(raw_df[col].nunique())
        print(raw_df[col].value_counts().head(15))

In [ ]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent
processed_folder = project_root / "data" / "processed"
output_folder = project_root / "outputs"
output_folder.mkdir(parents=True, exist_ok=True)

file_path = processed_folder / "thesouledstore_Oder_data_cleaned.csv"
sales_df = pd.read_csv(file_path, low_memory=False)

# Keep actual sales orders
sales_df = sales_df[
    sales_df["Order Classification"] == "SO"
].copy()

# Convert transaction dates into weekly periods
sales_df["Date"] = pd.to_datetime(
    sales_df["Date"], errors="coerce", dayfirst=True
)
sales_df = sales_df.dropna(subset=["Date", "Category1"])

sales_df["Week"] = (
    sales_df["Date"].dt.to_period("W-MON").dt.start_time
)

# Sum demand by week, location and category
weekly_df = (
    sales_df.groupby(
        ["Week", "Fulfilment Location", "Category1"],
        as_index=False
    )["Net Sold Quantity"]
    .sum()
)

# Fill missing weeks within each location-category history
filled_groups = []

for (loc, category), group in weekly_df.groupby(
    ["Fulfilment Location", "Category1"]
):
    group = group.sort_values("Week").set_index("Week")

    all_weeks = pd.date_range(
        group.index.min(), group.index.max(), freq="7D"
    )

    group = group.reindex(all_weeks)
    group["Fulfilment Location"] = loc
    group["Category1"] = category
    group.index.name = "Week"
    group["Net Sold Quantity"] = (
        group["Net Sold Quantity"].fillna(0)
    )

    filled_groups.append(group.reset_index())

category_df = pd.concat(filled_groups, ignore_index=True)

category_df.to_csv(
    processed_folder / "souledstore_category_weekly.csv",
    index=False
)

print("Dataset shape:", category_df.shape)
print("Locations:", category_df["Fulfilment Location"].nunique())
print("Categories:", category_df["Category1"].nunique())
print("Location-category combinations:",
      category_df.groupby(
          ["Fulfilment Location", "Category1"]
      ).ngroups)
print("Date range:", category_df["Week"].min(),
      "to", category_df["Week"].max())
print(category_df.head(10).to_string(index=False))

In [ ]:
import numpy as np

results = []

for (loc, category), group in category_df.groupby(
    ["Fulfilment Location", "Category1"]
):
    group = group.sort_values("Week")
    values = group["Net Sold Quantity"].to_numpy()

    if len(values) < 8:
        continue

    actual = values[-4:]
    train = values[:-4]

    naive_predictions = []
    average_predictions = []
    history = list(train)

    for i in range(4):
        naive_predictions.append(history[-1])
        average_predictions.append(np.mean(history[-4:]))
        history.append(actual[i])

    naive_mae = np.mean(np.abs(actual - naive_predictions))
    average_mae = np.mean(np.abs(actual - average_predictions))

    results.append({
        "Location": loc,
        "Category": category,
        "Naive MAE": naive_mae,
        "4-Week Average MAE": average_mae
    })

evaluation_df = pd.DataFrame(results)

print("Combinations evaluated:", len(evaluation_df))
print("\nAverage MAE:")
print(evaluation_df[["Naive MAE", "4-Week Average MAE"]].mean())

print("\nNaive wins:",
      (evaluation_df["Naive MAE"] < evaluation_df["4-Week Average MAE"]).sum())
print("4-week average wins:",
      (evaluation_df["4-Week Average MAE"] < evaluation_df["Naive MAE"]).sum())
print("Ties:",
      (evaluation_df["Naive MAE"] == evaluation_df["4-Week Average MAE"]).sum())

evaluation_df.to_csv(
    output_folder / "souledstore_category_model_evaluation.csv",
    index=False
)

In [ ]:
forecast_start = pd.to_datetime(category_df["Week"]).max() + pd.Timedelta(days=7)
future_weeks = pd.date_range(start=forecast_start, periods=4, freq="7D")

forecast_rows = []

for (loc, category), group in category_df.groupby(
    ["Fulfilment Location", "Category1"]
):
    group = group.sort_values("Week")
    recent_values = group["Net Sold Quantity"].tail(4).to_list()

    prediction = sum(recent_values) / len(recent_values)

    for week in future_weeks:
        forecast_rows.append({
            "Week": week,
            "Fulfilment Location": loc,
            "Category": category,
            "Forecast Net Sold Quantity": prediction
        })

category_forecast_df = pd.DataFrame(forecast_rows)

forecast_path = output_folder / "souledstore_category_4_week_forecast.csv"
category_forecast_df.to_csv(forecast_path, index=False)

print("Forecast rows:", len(category_forecast_df))
print("Unique location-category combinations:",
      category_forecast_df.groupby(
          ["Fulfilment Location", "Category"]
      ).ngroups)
print("Forecast weeks:", sorted(category_forecast_df["Week"].unique()))
print("\nFirst 10 rows:")
print(category_forecast_df.head(10).to_string(index=False))
print("\nSaved to:", forecast_path)